# 05 — Master Dataset Consolidation

This notebook consolidates all prepared data sources into a single, analysis-ready dataset for modeling.

The inputs include:

- ACEA car registration data (monthly, by country and power source)
- OECD macroeconomic indicators (e.g. CPI, interest rates, unemployment, consumer confidence)
- Industry-level indicators (e.g. vehicle parc, vehicle production)

The objective is to:

- Align all datasets on a common time granularity (monthly)
- Standardize country identifiers and formats
- Merge all inputs into a unified master dataset
- Prepare features required for downstream modeling

The final output is a structured dataset that serves as the foundation for the forecasting models developed in subsequent notebooks.

This step corresponds to the "gold" layer in a medallion-style data architecture, where all cleaned and enriched data sources are combined into a business-ready dataset.

In [ ]:
import pandas as pd
from functools import reduce
import os

In [ ]:
# =========================================================
# Load datasets
# =========================================================
fuel_monthly = pd.read_parquet("data/fuel_monthly.parquet")
Unemployment_df = pd.read_parquet("data/unemployment.parquet")
CCI_df = pd.read_parquet("data/cci.parquet")
CPI_df = pd.read_parquet("data/cpi.parquet")
gdp_monthly_df = pd.read_parquet("data/gdp_monthly.parquet")
rfc_monthly_df = pd.read_parquet("data/rfc_monthly.parquet")
interest_rates_df = pd.read_parquet("data/interest_rates.parquet")

# AND:
gd_auto_monthly_df = pd.read_parquet("data/gd_auto_monthly_df.parquet")
acea_df_all = pd.read_parquet("data/acea_df_all.parquet")

In [ ]:
# =========================================================
# 0. CHANING OCED INTEREST RATE DATASET COUNTRY NAME
# =========================================================

interest_rates_df_adj = interest_rates_df.copy()

interest_rates_df_adj["country"] = interest_rates_df_adj["country"].replace({"Euro area (19 countries)":"European Union (27 countries from 01/02/2020)"})
interest_rates_df_adj["country_code"] = interest_rates_df_adj["country_code"].replace({ "EA19":"EU27_2020"})

In [ ]:
display(interest_rates_df_adj["country_code"].unique())

In [ ]:
# =========================================================
# 0. CHANING OCED DATASETS COUNTRY CODES FROM 3 TO 2
# =========================================================

oecd_3_to_2 = {
    "AUT": "AT",
    "BEL": "BE",
    "BGR": "BG",
    "HRV": "HR",
    "CYP": "CY",
    "CZE": "CZ",
    "DNK": "DK",
    "EST": "EE",
    "FIN": "FI",
    "FRA": "FR",
    "DEU": "DE",
    "GRC": "GR",
    "HUN": "HU",
    "IRL": "IE",
    "ITA": "IT",
    "LVA": "LV",
    "LTU": "LT",
    "LUX": "LU",
    "MLT": "MT",
    "NLD": "NL",
    "POL": "PL",
    "PRT": "PT",
    "ROU": "RO",
    "SVK": "SK",
    "SVN": "SI",
    "ESP": "ES",
    "SWE": "SE",
    "ISL": "IS",
    "NOR": "NO",
    "CHE": "CH",
    "GBR": "UK",
    "EU27_2020": "EU",
    
}


oecd_datasets = {
    "Unemployment_df": Unemployment_df,
    "CCI_df": CCI_df,
    "CPI_df": CPI_df,
    "gdp_monthly_df": gdp_monthly_df,
    "rfc_monthly_df": rfc_monthly_df,
    "interest_rates_df_adj": interest_rates_df_adj,
}

for name, df in oecd_datasets.items():
    df["country_code"] = (
        df["country_code"]
        .astype(str)
        .str.strip()
        .replace(oecd_3_to_2)
    )

In [ ]:
# =========================================================
# FILTER OECD DATASETS TO ACEA COUNTRY CODES
# =========================================================

acea_country_codes = set(
    acea_df_all["country_code"].dropna().astype(str).str.strip().unique()
)

for name, df in oecd_datasets.items():
    oecd_datasets[name] = df[
        df["country_code"].astype(str).str.strip().isin(acea_country_codes)
    ].copy()

# reassign back
Unemployment_df = oecd_datasets["Unemployment_df"]
CCI_df = oecd_datasets["CCI_df"]
CPI_df = oecd_datasets["CPI_df"]
gdp_monthly_df = oecd_datasets["gdp_monthly_df"]
rfc_monthly_df = oecd_datasets["rfc_monthly_df"]
interest_rates_df_adj = oecd_datasets["interest_rates_df_adj"]

In [ ]:
# =========================================================
# 0. ENSURE DATE COLUMNS ARE DATETIME
# =========================================================
datasets_with_date = [
    ("fuel_monthly", fuel_monthly),
    ("Unemployment_df", Unemployment_df),
    ("CCI_df", CCI_df),
    ("CPI_df", CPI_df),
    ("gdp_monthly_df", gdp_monthly_df),
    ("rfc_monthly_df", rfc_monthly_df),
    ("interest_rates_df_adj", interest_rates_df),
    ("gd_auto_monthly_df", gd_auto_monthly_df),
    ("acea_df_all", acea_df_all),
]

for name, df in datasets_with_date:
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")


# =========================================================
# 1. STANDARDIZE EACH DATASET
# =========================================================

# -------------------------
# Fuel
# -------------------------
fuel_std = fuel_monthly.copy()

fuel_std = fuel_std[
    ["country", "country_code", "date", "year", "month", "quarter", "petrol_euro95", "petrol_euro95_yoy", "diesel"]
].copy()


# -------------------------
# Unemployment
# -------------------------
unemp_std = Unemployment_df.copy()

unemp_std = unemp_std.rename(columns={"value": "unemployment_rate"})

unemp_std = unemp_std[
    ["country", "country_code", "date", "year", "month", "quarter", "unemployment_rate"]
].copy()


# -------------------------
# Consumer Confidence
# -------------------------
cci_std = CCI_df.copy()

cci_std = cci_std.rename(columns={"value": "consumer_confidence"})

cci_std = cci_std[
    ["country", "country_code", "date", "year", "month", "quarter", "consumer_confidence"]
].copy()


# -------------------------
# CPI
# -------------------------
cpi_std = CPI_df.copy()

cpi_std = cpi_std.rename(columns={"value": "cpi"})

cpi_std = cpi_std[
    ["country", "country_code", "date", "year", "month", "quarter", "cpi"]
].copy()


# -------------------------
# GDP
# -------------------------
gdp_std = gdp_monthly_df.copy()

gdp_std = gdp_std.rename(columns={"value": "gdp_growth"})

gdp_std = gdp_std[
    ["country", "country_code", "date", "year", "month", "quarter", "gdp_growth"]
].copy()


# -------------------------
# Real Final Consumption
# -------------------------
rfc_std = rfc_monthly_df.copy()

rfc_std = rfc_std.rename(columns={"value": "real_final_consumption"})

rfc_std = rfc_std[
    ["country", "country_code", "date", "year", "month", "quarter", "real_final_consumption"]
].copy()


# -------------------------
# Interest Rates
# -------------------------
ir_std = interest_rates_df_adj.copy()

ir_std = ir_std.rename(columns={"value": "interest_rate"})

ir_std = ir_std[
    ["country", "country_code", "date", "year", "month", "quarter", "interest_rate"]
].copy()


# -------------------------
# GlobalData automotive
# -------------------------
gd_std = gd_auto_monthly_df.copy()

gd_std = gd_std[
    ["country", "country_code", "date", "year", "month", "quarter", "vehicle_parc", "vehicle_production", "vehicle_parc_yoy"]
].copy()


# -------------------------
# ACEA registrations / fuel mix
# Rename columns to avoid collision with fuel price columns
# -------------------------
acea_std = acea_df_all.copy()

acea_std = acea_std.rename(columns={
    "battery_electric": "acea_battery_electric",
    "plug_in_hybrid": "acea_plug_in_hybrid",
    "hybrid_electric": "acea_hybrid_electric",
    "others": "acea_others",
    "petrol": "acea_petrol_registrations",
    "diesel": "acea_diesel_registrations",
    "total": "acea_total_registrations"
})

# If date does not exist in acea_df_all, reconstruct it from year/month
if "date" not in acea_std.columns:
    acea_std["date"] = pd.to_datetime(
        dict(year=acea_std["year"], month=acea_std["month"], day=1),
        errors="coerce"
    )

acea_std = acea_std[
    [
        "country",
        "country_code",
        "date",
        "year",
        "month",
        "quarter",
        "acea_battery_electric",
        "acea_plug_in_hybrid",
        "acea_hybrid_electric",
        "acea_others",
        "acea_petrol_registrations",
        "acea_diesel_registrations",
        "acea_total_registrations",
        "extraction_method"
    ]
].copy()


# =========================================================
# 2. OPTIONAL: CLEAN DUPLICATES IN EACH DATASET
# =========================================================
def deduplicate_monthly(df, name):
    dupes = df.duplicated(subset=["country_code", "year", "month"]).sum()
    print(f"{name}: duplicates on (country_code, year, month) = {dupes}")
    if dupes > 0:
        df = df.drop_duplicates(subset=["country_code", "year", "month"]).copy()
    return df

fuel_std = deduplicate_monthly(fuel_std, "fuel_std")
unemp_std = deduplicate_monthly(unemp_std, "unemp_std")
cci_std = deduplicate_monthly(cci_std, "cci_std")
cpi_std = deduplicate_monthly(cpi_std, "cpi_std")
gdp_std = deduplicate_monthly(gdp_std, "gdp_std")
rfc_std = deduplicate_monthly(rfc_std, "rfc_std")
ir_std = deduplicate_monthly(ir_std, "ir_std")
gd_std = deduplicate_monthly(gd_std, "gd_std")
acea_std = deduplicate_monthly(acea_std, "acea_std")


# =========================================================
# 3. CHOOSE A BASE DATAFRAME
# =========================================================
# Base is unemployment because it contains:
# country, country_code, date, year, month, quarter
base_df = unemp_std.copy()


# =========================================================
# 4. MERGE EVERYTHING
# =========================================================
dfs_to_merge = [
    base_df,
    cci_std.drop(columns=["country", "date", "quarter"], errors="ignore"),
    cpi_std.drop(columns=["country", "date", "quarter"], errors="ignore"),
    gdp_std.drop(columns=["country", "date", "quarter"], errors="ignore"),
    rfc_std.drop(columns=["country", "date", "quarter"], errors="ignore"),
    ir_std.drop(columns=["country", "date", "quarter"], errors="ignore"),
    fuel_std.drop(columns=["country", "date", "quarter"], errors="ignore"),
    gd_std.drop(columns=["country", "date", "quarter"], errors="ignore"),
    acea_std.drop(columns=["country", "date", "quarter"], errors="ignore"),
]

master_df = reduce(
    lambda left, right: pd.merge(
        left, right, on=["country_code", "year", "month"], how="left"
    ),
    dfs_to_merge
)


# =========================================================
# 5. REBUILD TIME FIELDS FROM DATE
# =========================================================
master_df["date"] = pd.to_datetime(
    dict(year=master_df["year"], month=master_df["month"], day=1),
    errors="coerce"
)
master_df["quarter"] = master_df["date"].dt.quarter


# =========================================================
# 6. REORDER COLUMNS
# =========================================================
desired_order = [
    "country",
    "country_code",
    "date",
    "year",
    "month",
    "quarter",
    "unemployment_rate",
    "consumer_confidence",
    "cpi",
    "gdp_growth",
    "real_final_consumption",
    "interest_rate",
    "petrol_euro95",
    "petrol_euro95_yoy",
    "diesel",
    "vehicle_parc",
    "vehicle_parc_yoy", 
    "vehicle_production",
    "acea_battery_electric",
    "acea_plug_in_hybrid",
    "acea_hybrid_electric",
    "acea_others",
    "acea_petrol_registrations",
    "acea_diesel_registrations",
    "acea_total_registrations",
    "extraction_method",
]

existing_order = [c for c in desired_order if c in master_df.columns]
remaining_cols = [c for c in master_df.columns if c not in existing_order]

master_df = master_df[existing_order + remaining_cols]



# =========================================================
# 7. SORT
# =========================================================
master_df = master_df.sort_values(["country_code", "date"]).reset_index(drop=True)


# =========================================================
# 8. VALIDATION CHECKS
# =========================================================
print("Shape:", master_df.shape)
print("Duplicates on (country, date):", master_df.duplicated(subset=["country", "date"]).sum())

print("\nMissing values per column:")
print(master_df.isna().sum())

display(master_df)

# Impute some of the missing data for gdp and rfc

In [ ]:
master_df_impute = master_df.copy()

master_df_impute = master_df_impute.sort_values(["country_code", "date"])

master_df_impute["gdp_growth"] = (
    master_df_impute.groupby("country_code")["gdp_growth"].ffill()
)

master_df_impute["real_final_consumption"] = (
    master_df_impute.groupby("country_code")["real_final_consumption"].ffill()
)


# Countries that are excluded due to fully missing data, either vehicle parc, gdp, rfc or cpi (or sometimes several variables)

In [ ]:
countries_no_gd = ["BG", "EE", "HR", "LT", "LU", "LV", "RO", "UK"]

master_df_impute = master_df_impute[
    ~master_df_impute["country_code"].isin(countries_no_gd)
].copy()

In [ ]:
print("\nMissing values per column:")
print(master_df_impute.isna().sum())

In [ ]:
display(master_df_impute[master_df_impute["country_code"] == "EU"] [["country", "country_code", "interest_rate", "date"]])

In [ ]:
# =========================================================
# Change EU country name to simplify
# =========================================================

master_df_impute["country"] = master_df_impute["country"].replace({"European Union (27 countries from 01/02/2020)":"European Union"})


In [ ]:
# =========================================================
# Dropping unnecessary columns
# =========================================================


master_df_impute = master_df_impute.drop(
    columns=["gdp_key_exists", "rfc_key_exists", "acea_petrol_registrations", "acea_diesel_registrations", "acea_hybrid_electric", "acea_plug_in_hybrid", "acea_others"],
    errors="ignore"
)

In [ ]:
# =========================================================
# adding some final columns
# =========================================================


import numpy as np

# ratio version (better for modeling)
master_df_impute["ev_adoption_ratio"] = (
    master_df_impute["acea_battery_electric"] / 
    master_df_impute["acea_total_registrations"].replace(0, np.nan)
)

# percentage version (better for interpretation / plots)
master_df_impute["ev_adoption_pct"] = master_df_impute["ev_adoption_ratio"] * 100

In [ ]:
display(
    master_df_impute[
        (master_df_impute["country_code"] == "EU") &
        ((master_df_impute["year"] == 2025))
    ]
    .sort_values(["country_code", "date"])
    .groupby("country_code")[["cpi", "unemployment_rate", "country_code", "date"]]
    .head(12)
)

In [ ]:
# =========================================================
#  Saving dataframe to desk 
# =========================================================


# Create folder if it doesn't exist
os.makedirs("data", exist_ok=True)

# Save datasets
master_df_impute.to_parquet("data/master_df_impute.parquet", index=False)
